# Week 1 — Data Understanding & Profiling

## Objective

The purpose of this notebook is to understand the structure, quality, and relationships of the CodeTX supply-chain datasets before performing deeper analysis.

## Week 1 Goals

- Load all project datasets
- Review dataset dimensions
- Inspect column names and data types
- Identify missing values
- Check duplicate records
- Identify possible primary and foreign keys
- Understand relationships between tables
- Document initial data quality issues

In [1]:
import pandas as pd
import numpy as np

In [2]:
branches = pd.read_csv("branches.csv")
customers = pd.read_csv("customers.csv")
inventory = pd.read_csv("inventory_master.csv")
invoices = pd.read_csv("invoices.csv")
payments = pd.read_csv("payments.csv")
products = pd.read_csv("products.csv")
po_header = pd.read_csv("purchase_orders_header.csv")
po_lines = pd.read_csv("purchase_orders_lines.csv")
so_header = pd.read_csv("sales_orders_header.csv")
so_lines = pd.read_csv("sales_orders_lines.csv")
stock_ledger = pd.read_csv("stock_ledger.csv")
suppliers = pd.read_csv("suppliers.csv")

In [3]:
datasets = {
    "branches": branches,
    "customers": customers,
    "inventory": inventory,
    "invoices": invoices,
    "payments": payments,
    "products": products,
    "po_header": po_header,
    "po_lines": po_lines,
    "so_header": so_header,
    "so_lines": so_lines,
    "stock_ledger": stock_ledger,
    "suppliers": suppliers
}

In [4]:
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

branches: (6, 13)
customers: (500, 15)
inventory: (180, 8)
invoices: (18033, 10)
payments: (19257, 5)
products: (30, 23)
po_header: (24000, 10)
po_lines: (155495, 9)
so_header: (20000, 11)
so_lines: (130402, 9)
stock_ledger: (237230, 9)
suppliers: (8, 12)


### Initial Dataset Structure

The project contains 12 relational datasets covering branches, customers, products, suppliers, inventory, procurement, sales, invoices, payments, and stock movements.

The smaller datasets such as `branches`, `suppliers`, `products`, and `customers` primarily represent master or reference data.

The larger datasets such as `sales_orders_lines`, `purchase_orders_lines`, and `stock_ledger` contain transactional records.

The `stock_ledger` is the largest dataset with 237,230 records and is expected to be particularly important for inventory movement, demand analysis, and time-based analysis.

The `inventory` table contains 180 rows. Since there are 30 products and 6 branches, this may represent one inventory record for every product-branch combination. This relationship will be validated later.

In [5]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("-" * 60)

    profile = pd.DataFrame({
        "data_type": df.dtypes,
        "missing_values": df.isnull().sum(),
        "missing_percent": (df.isnull().sum() / len(df) * 100).round(2)
    })

    display(profile)


BRANCHES
------------------------------------------------------------


,data_type,missing_values,missing_percent
branch_id,str,0,0.0
branch_name,str,0,0.0
city,str,0,0.0
state,str,0,0.0
region,str,0,0.0
warehouse_type,str,0,0.0
warehouse_capacity,str,0,0.0
service_center_available,str,0,0.0
manager_id,int64,0,0.0
total_employees,int64,0,0.0



CUSTOMERS
------------------------------------------------------------


,data_type,missing_values,missing_percent
customer_id,str,0,0.0
customer_type,str,0,0.0
industry_segment,str,0,0.0
city,str,0,0.0
state,str,0,0.0
pincode,int64,0,0.0
region,str,0,0.0
branch_id,str,0,0.0
credit_limit,int64,0,0.0
current_balance,int64,0,0.0



INVENTORY
------------------------------------------------------------


...

,data_type,missing_values,missing_percent
so_id,str,0,0.0
line_number,int64,0,0.0
product_id,str,0,0.0
quantity,int64,0,0.0
unit_price,int64,0,0.0
gst_rate,int64,0,0.0
line_total,int64,0,0.0
gst_amount,float64,0,0.0
line_grand_total,float64,0,0.0



STOCK_LEDGER
------------------------------------------------------------


,data_type,missing_values,missing_percent
movement_id,str,0,0.0
product_id,str,0,0.0
branch_id,str,0,0.0
movement_type,str,0,0.0
movement_date,str,0,0.0
quantity,int64,0,0.0
reference_type,str,0,0.0
reference_id,str,0,0.0
running_balance,int64,0,0.0



SUPPLIERS
------------------------------------------------------------


,data_type,missing_values,missing_percent
supplier_id,str,0,0.0
supplier_name,str,0,0.0
supplier_type,str,0,0.0
product_category,str,0,0.0
city,str,0,0.0
province,str,0,0.0
region,str,0,0.0
pincode,int64,0,0.0
lead_time_days,int64,0,0.0
reliability_score,int64,0,0.0


In [6]:
key_checks = {
    "branches": "branch_id",
    "customers": "customer_id",
    "products": "product_id",
    "suppliers": "supplier_id",
    "invoices": "invoice_id",
    "payments": "payment_id",
    "po_header": "po_id",
    "so_header": "so_id",
    "stock_ledger": "movement_id"
}

for name, key in key_checks.items():
    df = datasets[name]

    print(f"\n{name.upper()}")
    print("-" * 50)
    print("Rows:", len(df))
    print("Unique IDs:", df[key].nunique())
    print("Duplicate IDs:", df[key].duplicated().sum())


BRANCHES
--------------------------------------------------
Rows: 6
Unique IDs: 6
Duplicate IDs: 0

CUSTOMERS
--------------------------------------------------
Rows: 500
Unique IDs: 500
Duplicate IDs: 0

PRODUCTS
--------------------------------------------------
Rows: 30
Unique IDs: 30
Duplicate IDs: 0

SUPPLIERS
--------------------------------------------------
Rows: 8
Unique IDs: 8
Duplicate IDs: 0

INVOICES
--------------------------------------------------
Rows: 18033
Unique IDs: 17836
Duplicate IDs: 197

PAYMENTS
--------------------------------------------------
Rows: 19257
Unique IDs: 19055
Duplicate IDs: 202

PO_HEADER
--------------------------------------------------
Rows: 24000
Unique IDs: 24000
Duplicate IDs: 0

SO_HEADER
--------------------------------------------------
Rows: 20000
Unique IDs: 20000
Duplicate IDs: 0

STOCK_LEDGER
--------------------------------------------------
Rows: 237230
Unique IDs: 237230
Duplicate IDs: 0


In [7]:
invoice_duplicates = invoices[
    invoices["invoice_id"].duplicated(keep=False)
].sort_values("invoice_id")

payment_duplicates = payments[
    payments["payment_id"].duplicated(keep=False)
].sort_values("payment_id")

print("Invoice duplicate rows:")
display(invoice_duplicates.head(20))

print("\nPayment duplicate rows:")
display(payment_duplicates.head(20))

Invoice duplicate rows:


,invoice_id,so_id,customer_id,branch_id,invoice_date,due_date,total_order_value,total_gst_amount,grand_total,payment_status
14917,INV-105711,SO-901766,C0208,KOL001,2021-11-20,2022-01-04,1424270,368413.6,1792683.6,Paid
8324,INV-105711,SO-549003,C0191,KOL001,2024-12-15,2025-01-14,529940,146609.2,676549.2,Paid
14643,INV-106596,SO-807417,C0281,KOL001,2024-01-25,2024-03-25,40100,7218.0,47318.0,Paid
9894,INV-106596,SO-399413,C0409,PUN001,2021-06-16,2021-07-31,1204730,336001.4,1540731.4,Paid
1712,INV-111054,SO-233671,C0072,PUN001,2023-04-11,2023-04-26,1207780,319950.4,1527730.4,Paid
1706,INV-111054,SO-316263,C0313,AHM001,2019-08-02,2019-10-01,310,55.8,365.8,Paid
13853,INV-112235,SO-236576,C0220,KOL001,2021-04-17,2021-06-16,1479960,405362.8,1885322.8,Paid
946,INV-112235,SO-363636,C0198,AHM001,2021-01-08,2021-03-09,2369250,654525.0,3023775.0,Paid
13492,INV-112378,SO-911241,C0082,AHM001,2025-01-01,2025-03-02,3833070,1071302.6,4904372.6,Paid
1609,INV-112378,SO-870768,C0104,KOL001,2019-04-15,2019-06-14,2036600,570248.0,2606848.0,Paid



Payment duplicate rows:


,payment_id,invoice_id,payment_date,payment_amount,payment_method
19220,PAY-102690,INV-826669,2021-04-23,3837051.40,Credit Card
7121,PAY-102690,INV-196859,2021-12-29,520518.80,Cash
16981,PAY-105592,INV-963449,2023-02-23,2123180.75,Cash
11803,PAY-105592,INV-323176,2022-07-10,1019902.14,Cheque
15115,PAY-112476,INV-743633,2024-06-17,4544341.00,Credit Card
5761,PAY-112476,INV-374653,2024-05-16,1724913.40,Cash
1148,PAY-115474,INV-645256,2022-12-23,473838.80,Credit Card
3929,PAY-115474,INV-583030,2023-08-25,3764707.00,Credit Card
5117,PAY-115921,INV-301855,2022-01-06,353798.70,UPI
9009,PAY-115921,INV-121000,2021-05-08,246614.08,Cash


### Duplicate ID Investigation

The duplicated `invoice_id` and `payment_id` values were reviewed in detail.

The repeated IDs do not represent identical duplicate rows. Instead, the same identifier is assigned to different transactions with different related records, dates, customers, amounts, and payment details.

This indicates an identifier integrity issue rather than a simple row duplication problem.

No rows were removed at this stage because deleting them without understanding the relationship impact could cause loss of valid transaction records.

A separate strategy will be developed later for preserving the records while ensuring reliable joins and analysis.

In [8]:
for name, df in datasets.items():
    exact_duplicates = df.duplicated().sum()
    print(f"{name}: {exact_duplicates} exact duplicate rows")

branches: 0 exact duplicate rows
customers: 0 exact duplicate rows
inventory: 0 exact duplicate rows
invoices: 0 exact duplicate rows
payments: 0 exact duplicate rows
products: 0 exact duplicate rows
po_header: 0 exact duplicate rows
po_lines: 0 exact duplicate rows
so_header: 0 exact duplicate rows
so_lines: 0 exact duplicate rows
stock_ledger: 0 exact duplicate rows
suppliers: 0 exact duplicate rows


In [9]:
relationship_checks = [
    ("customers.branch_id", customers["branch_id"], branches["branch_id"]),
    ("inventory.product_id", inventory["product_id"], products["product_id"]),
    ("inventory.branch_id", inventory["branch_id"], branches["branch_id"]),
    ("invoices.so_id", invoices["so_id"], so_header["so_id"]),
    ("invoices.customer_id", invoices["customer_id"], customers["customer_id"]),
    ("invoices.branch_id", invoices["branch_id"], branches["branch_id"]),
    ("payments.invoice_id", payments["invoice_id"], invoices["invoice_id"]),
    ("po_header.supplier_id", po_header["supplier_id"], suppliers["supplier_id"]),
    ("po_header.branch_id", po_header["branch_id"], branches["branch_id"]),
    ("po_lines.po_id", po_lines["po_id"], po_header["po_id"]),
    ("po_lines.product_id", po_lines["product_id"], products["product_id"]),
    ("so_header.customer_id", so_header["customer_id"], customers["customer_id"]),
    ("so_header.branch_id", so_header["branch_id"], branches["branch_id"]),
    ("so_lines.so_id", so_lines["so_id"], so_header["so_id"]),
    ("so_lines.product_id", so_lines["product_id"], products["product_id"]),
    ("stock_ledger.product_id", stock_ledger["product_id"], products["product_id"]),
    ("stock_ledger.branch_id", stock_ledger["branch_id"], branches["branch_id"])
]

for relationship, child_values, parent_values in relationship_checks:
    invalid_values = ~child_values.isin(parent_values)
    invalid_count = invalid_values.sum()
    print(f"{relationship}: {invalid_count} unmatched values")

customers.branch_id: 0 unmatched values
inventory.product_id: 0 unmatched values
inventory.branch_id: 0 unmatched values
invoices.so_id: 0 unmatched values
invoices.customer_id: 0 unmatched values
invoices.branch_id: 0 unmatched values
payments.invoice_id: 0 unmatched values
po_header.supplier_id: 0 unmatched values
po_header.branch_id: 0 unmatched values
po_lines.po_id: 0 unmatched values
po_lines.product_id: 0 unmatched values
so_header.customer_id: 0 unmatched values
so_header.branch_id: 0 unmatched values
so_lines.so_id: 0 unmatched values
so_lines.product_id: 0 unmatched values
stock_ledger.product_id: 0 unmatched values
stock_ledger.branch_id: 0 unmatched values


In [10]:
date_columns = {
    "customers": ["last_purchase_date"],
    "invoices": ["invoice_date", "due_date"],
    "payments": ["payment_date"],
    "products": ["last_purchase_date"],
    "po_header": ["order_date", "expected_delivery_date", "received_date"],
    "so_header": ["order_date", "delivery_date"],
    "stock_ledger": ["movement_date"]
}

for dataset_name, columns in date_columns.items():
    print(f"\n{dataset_name.upper()}")
    print("-" * 60)

    df = datasets[dataset_name]

    for column in columns:
        converted = pd.to_datetime(df[column], errors="coerce")

        print(
            f"{column}: "
            f"{converted.min().date()} to {converted.max().date()} | "
            f"Missing/invalid: {converted.isna().sum()}"
        )


CUSTOMERS
------------------------------------------------------------
last_purchase_date: 2015-04-30 to 2024-12-27 | Missing/invalid: 0

INVOICES
------------------------------------------------------------
invoice_date: 2019-01-03 to 2025-01-13 | Missing/invalid: 0
due_date: 2019-01-20 to 2025-03-14 | Missing/invalid: 0

PAYMENTS
------------------------------------------------------------
payment_date: 2019-01-09 to 2025-03-20 | Missing/invalid: 0

PRODUCTS
------------------------------------------------------------
last_purchase_date: 2019-03-15 to 2025-04-27 | Missing/invalid: 0

PO_HEADER
------------------------------------------------------------
order_date: 2019-01-01 to 2024-12-31 | Missing/invalid: 0
expected_delivery_date: 2019-01-11 to 2025-01-28 | Missing/invalid: 0
received_date: 2019-01-11 to 2025-01-28 | Missing/invalid: 2370

SO_HEADER
------------------------------------------------------------
order_date: 2019-01-01 to 2024-12-31 | Missing/invalid: 0
delivery_date

In [11]:
po_missing_received = po_header[
    po_header["received_date"].isna()
]

po_missing_received["po_status"].value_counts(dropna=False)

po_status
Cancelled    2370
Name: count, dtype: int64

In [12]:
pd.crosstab(
    po_header["po_status"],
    po_header["received_date"].isna(),
    margins=True
)

received_date,False,True,All
po_status,,,
Cancelled,0,2370,2370
Received,21630,0,21630
All,21630,2370,24000


### Missing `received_date` Investigation

The `received_date` column contains 2,370 missing values.

After comparing the missing values with `po_status`, all 2,370 missing dates were found to belong to cancelled purchase orders. All 21,630 orders with status `Received` contain a valid `received_date`.

Therefore, the missing values are logically consistent with the business process and do not represent a data-quality error.

The missing `received_date` values will be preserved rather than imputed or removed.

In [24]:
date_columns = {
    "customers": ["customer_since", "last_purchase_date"],
    "invoices": ["invoice_date", "due_date"],
    "payments": ["payment_date"],
    "products": ["last_purchase_date"],
    "po_header": ["order_date", "expected_delivery_date", "received_date"],
    "so_header": ["order_date", "delivery_date"],
    "stock_ledger": ["movement_date"]
}

for dataset_name, columns in date_columns.items():
    for column in columns:
        datasets[dataset_name][column] = pd.to_datetime(
            datasets[dataset_name][column],
            errors="coerce"
        )

In [14]:
for dataset_name, columns in date_columns.items():
    print(f"\n{dataset_name.upper()}")

    for column in columns:
        print(f"{column}: {datasets[dataset_name][column].dtype}")


CUSTOMERS
customer_since: datetime64[us]
last_purchase_date: datetime64[us]

INVOICES
invoice_date: datetime64[us]
due_date: datetime64[us]

PAYMENTS
payment_date: datetime64[us]

PRODUCTS
last_purchase_date: datetime64[us]

PO_HEADER
order_date: datetime64[us]
expected_delivery_date: datetime64[us]
received_date: datetime64[us]

SO_HEADER
order_date: datetime64[us]
delivery_date: datetime64[us]

STOCK_LEDGER
movement_date: datetime64[us]


In [15]:
inventory_check = inventory.groupby(
    ["product_id", "branch_id"]
).size().reset_index(name="count")

print("Total inventory rows:", len(inventory))
print("Unique product-branch combinations:", len(inventory_check))
print("Duplicate product-branch combinations:",
      (inventory_check["count"] > 1).sum())

print("Unique products in inventory:",
      inventory["product_id"].nunique())

print("Unique branches in inventory:",
      inventory["branch_id"].nunique())

print("Expected combinations:",
      inventory["product_id"].nunique() * inventory["branch_id"].nunique())

Total inventory rows: 180
Unique product-branch combinations: 180
Duplicate product-branch combinations: 0
Unique products in inventory: 30
Unique branches in inventory: 6
Expected combinations: 180


In [16]:
branches["warehouse_capacity"].head(10)

0    45230 sqft
1    38750 sqft
2    41890 sqft
3    36210 sqft
4    29880 sqft
5    34120 sqft
Name: warehouse_capacity, dtype: str

In [17]:
branches["warehouse_capacity_sqft"] = (
    branches["warehouse_capacity"]
    .str.replace(" sqft", "", regex=False)
    .astype(int)
)

In [18]:
branches[["warehouse_capacity", "warehouse_capacity_sqft"]]

,warehouse_capacity,warehouse_capacity_sqft
0,45230 sqft,45230
1,38750 sqft,38750
2,41890 sqft,41890
3,36210 sqft,36210
4,29880 sqft,29880
5,34120 sqft,34120


In [19]:
numeric_checks = {
    "branches": [
        "warehouse_capacity_sqft",
        "total_employees",
        "avg_monthly_revenue",
        "monthly_operational_cost",
        "market_demand_index"
    ],
    "customers": [
        "credit_limit",
        "current_balance",
        "total_purchase_value",
        "customer_rating"
    ],
    "inventory": [
        "opening_stock",
        "reorder_level",
        "safety_stock",
        "max_stock",
        "current_stock"
    ],
    "products": [
        "unit_cost",
        "unit_price",
        "margin_percentage",
        "gst_rate",
        "weight_kg",
        "warranty_months",
        "reorder_level",
        "safety_stock",
        "max_stock_level",
        "lead_time_days",
        "last_purchase_price"
    ],
    "po_header": [
        "total_cost",
        "total_gst_amount",
        "grand_total"
    ],
    "po_lines": [
        "quantity",
        "unit_cost",
        "gst_rate",
        "line_total",
        "gst_amount",
        "line_grand_total"
    ],
    "so_header": [
        "total_order_value",
        "total_gst_amount",
        "grand_total"
    ],
    "so_lines": [
        "quantity",
        "unit_price",
        "gst_rate",
        "line_total",
        "gst_amount",
        "line_grand_total"
    ],
    "payments": [
        "payment_amount"
    ],
    "stock_ledger": [
        "quantity",
        "running_balance"
    ],
    "suppliers": [
        "lead_time_days",
        "reliability_score",
        "import_duty_rate"
    ]
}

for dataset_name, columns in numeric_checks.items():
    print(f"\n{dataset_name.upper()}")
    print("-" * 60)

    df = datasets[dataset_name]

    for column in columns:
        print(
            f"{column}: "
            f"min={df[column].min()}, "
            f"max={df[column].max()}, "
            f"negative={(df[column] < 0).sum()}, "
            f"zero={(df[column] == 0).sum()}"
        )


BRANCHES
------------------------------------------------------------
warehouse_capacity_sqft: min=29880, max=45230, negative=0, zero=0
total_employees: min=49, max=92, negative=0, zero=0
avg_monthly_revenue: min=2894770, max=5234890, negative=0, zero=0
monthly_operational_cost: min=1743000, max=3189000, negative=0, zero=0
market_demand_index: min=6, max=9, negative=0, zero=0

CUSTOMERS
------------------------------------------------------------
credit_limit: min=10552, max=1493816, negative=0, zero=0
current_balance: min=63, max=1435105, negative=0, zero=0
total_purchase_value: min=5173, max=4998438, negative=0, zero=0
customer_rating: min=1, max=5, negative=0, zero=0

INVENTORY
------------------------------------------------------------
opening_stock: min=80, max=300, negative=0, zero=0
reorder_level: min=24, max=119, negative=0, zero=0
safety_stock: min=16, max=85, negative=0, zero=0
max_stock: min=131, max=582, negative=0, zero=0
current_stock: min=88853, max=121013, negative=0,

In [20]:
for dataset_name, columns in numeric_checks.items():
    df = datasets[dataset_name]

    for column in columns:
        if not pd.api.types.is_numeric_dtype(df[column]):
            print(
                f"{dataset_name}.{column} -> "
                f"{df[column].dtype}"
            )

In [21]:
stock_ledger["movement_type"].value_counts()

movement_type
IN            127611
OUT           107165
ADJUSTMENT      2454
Name: count, dtype: int64

In [22]:
latest_stock = (
    stock_ledger
    .sort_values("movement_date")
    .groupby(["product_id", "branch_id"])
    .tail(1)
)

stock_validation = inventory.merge(
    latest_stock[
        ["product_id", "branch_id", "running_balance", "movement_date"]
    ],
    on=["product_id", "branch_id"],
    how="left"
)

stock_validation["difference"] = (
    stock_validation["current_stock"]
    - stock_validation["running_balance"]
)

print("Rows checked:", len(stock_validation))
print(
    "Exact matches:",
    (stock_validation["difference"] == 0).sum()
)

print(
    "Non-matches:",
    (stock_validation["difference"] != 0).sum()
)

display(
    stock_validation[
        [
            "product_id",
            "branch_id",
            "current_stock",
            "running_balance",
            "difference"
        ]
    ].head(10)
)

Rows checked: 180
Exact matches: 180
Non-matches: 0


,product_id,branch_id,current_stock,running_balance,difference
0,P001,DEL001,115928,115928,0
1,P001,PUN001,109389,109389,0
2,P001,CHN001,99373,99373,0
3,P001,HYD001,108772,108772,0
4,P001,KOL001,103808,103808,0
5,P001,AHM001,106714,106714,0
6,P002,DEL001,108628,108628,0
7,P002,PUN001,101623,101623,0
8,P002,CHN001,105885,105885,0
9,P002,HYD001,121013,121013,0


### Inventory Balance Validation

The `stock_ledger` stores movement quantities as positive values, while the direction of movement is represented separately through `movement_type`, such as `IN`, `OUT`, and `ADJUSTMENT`.

The unusually large values in `inventory.current_stock` were validated against the latest `running_balance` in the stock ledger for every product-branch combination.

All 180 inventory records matched their corresponding latest ledger balance exactly.

Therefore, the `current_stock` values are internally consistent with the stock movement history and should not be treated as corrupted values solely because they are much larger than the opening stock or inventory threshold fields.

In [23]:
categorical_checks = {
    "branches": [
        "region",
        "warehouse_type",
        "service_center_available"
    ],
    "customers": [
        "customer_type",
        "industry_segment",
        "region",
        "payment_terms"
    ],
    "products": [
        "category",
        "machine_type",
        "criticality_level",
        "usage_frequency",
        "uom"
    ],
    "suppliers": [
        "supplier_type",
        "product_category",
        "region"
    ],
    "po_header": [
        "po_status"
    ],
    "so_header": [
        "order_status",
        "payment_terms",
        "sales_channel"
    ],
    "invoices": [
        "payment_status"
    ],
    "payments": [
        "payment_method"
    ],
    "stock_ledger": [
        "movement_type",
        "reference_type"
    ]
}

for dataset_name, columns in categorical_checks.items():
    print(f"\n{dataset_name.upper()}")
    print("-" * 60)

    df = datasets[dataset_name]

    for column in columns:
        print(f"\n{column}")
        print(df[column].value_counts(dropna=False))


BRANCHES
------------------------------------------------------------

region
region
West     2
South    2
North    1
East     1
Name: count, dtype: int64

warehouse_type
warehouse_type
Regional       2
Branch         2
Central        1
Service Hub    1
Name: count, dtype: int64

service_center_available
service_center_available
Yes    4
No     2
Name: count, dtype: int64

CUSTOMERS
------------------------------------------------------------

customer_type
customer_type
Government     114
Corporate      104
Dealer          99
Fleet Owner     95
Retail          88
Name: count, dtype: int64

industry_segment
industry_segment
Mining            98
Logistics         96
Construction      90
Infrastructure    79
Manufacturing     70
Rental            67
Name: count, dtype: int64

region
region
South         158
West          151
East           87
North          82
Central        11
North East     11
Name: count, dtype: int64

payment_terms
payment_terms
Net 45     113
Net 60     107
Advance

### Categorical Data Quality Summary

The major categorical variables were reviewed for inconsistent spelling, capitalization, unexpected labels, and missing categories.

The values appear internally consistent across the reviewed fields.

Key observations include:

- Purchase orders are classified as either `Received` or `Cancelled`.
- Sales orders are classified as either `Delivered` or `Cancelled`.
- Invoice payment status includes `Paid`, `Partially Paid`, and `Unpaid`.
- Stock movements are classified as `IN`, `OUT`, or `ADJUSTMENT`.
- Stock ledger reference types correspond logically to purchase orders (`PO`), sales orders (`SO`), and adjustments (`ADJ`).
- Customer, product, supplier, warehouse, payment, and sales-channel categories appear consistently formatted.

No major categorical standardization issues were identified during the initial profiling stage.

# Week 1 — Data Quality Summary

## Key Findings

1. All 12 project datasets were successfully loaded and reviewed.

2. The dataset follows a relational structure linking branches, products, customers, suppliers, purchasing, sales, invoices, payments, inventory, and stock movements.

3. No exact duplicate rows were found across the 12 datasets.

4. Duplicate identifier values were found in `invoice_id` and `payment_id`. These duplicated IDs belong to different transactions and therefore represent an identifier-integrity issue rather than simple row duplication.

5. Foreign-key relationships between the major tables were validated and no unmatched reference values were identified.

6. The 2,370 missing values in `purchase_orders_header.received_date` correspond entirely to cancelled purchase orders. These missing values are therefore logically valid and should not be imputed or removed.

7. Date fields were converted to proper datetime format for later time-based analysis.

8. The `inventory_master` table contains exactly one record for each of the 30 products across the 6 branches, producing 180 unique product-branch combinations.

9. `warehouse_capacity` was originally stored as text with the `sqft` unit. A numeric `warehouse_capacity_sqft` field was created for analysis while preserving the original field.

10. Inventory `current_stock` values were validated against the latest stock-ledger running balances. All 180 product-branch records matched exactly.

11. Stock ledger quantities are stored as positive values, while movement direction is represented by the `movement_type` field.

12. Major categorical fields were reviewed and no significant spelling, capitalization, or category-consistency issues were identified.

## Week 1 Conclusion

The datasets are generally well structured and suitable for further analysis.

The main issue requiring special handling is the duplicated invoice and payment identifiers. These records should not be deleted automatically because they represent different transactions.

The project is ready to move into the next phase: structured data cleaning, analytical feature preparation, and integration.